# Compare Dynamic Panel Training Windows (Validation Only)

Next step after Notebook 10: compare 30, 60, 90 and 180 training-origin days on exactly the same NJ regions and 2021 validation dates. Keep horizons 1/2/3, inputs and weekly refitting fixed. Compare with/without pressure at each window. No test forecasts are generated here.

Dependencies: numpy, pandas. Run from Weikun. Reads `nj_daily_feature_outputs/nj_daily_features.csv.gz` and saves into `dynamic_panel_window_comparison_outputs` under Weikun. No source data, feature files or previous model results are overwritten.


## Decision rule and interpretation

The primary candidate is Dynamic Panel + 7-day pressure. Select its window using the smallest mean validation MAE across days 1/2/3. RMSE then shorter window break ties. Also report a separate ranking for the no-pressure model. To isolate the pressure contribution, compare the two models at the SAME window, cohort and dates, not each at its own best setting.

Window choices are conditional on this model, features and validation period. They do not prove that a window is universally best. Weekly parameter refits and daily input updates follow Notebook 10. All training targets occur on or before each fit date. Different windows may use different training history, but evaluation rows are asserted identical.

This remains retrospective recorded-onset evaluation. Onset completeness, invalid origin-population treatment and unvalidated ZIP/ZCTA correspondence retain their earlier limitations. Earlier test results have already been viewed; do not describe a later assessment on those dates as untouched new evidence.


## 1. Functions

Self-contained fixed-effects OLS and validation-only experiment functions.

In [1]:
"""Retrospective fixed-effects OLS baseline for direct 1/2/3-day onset-count forecasts.

Dependencies: numpy, pandas. No GMM, causal estimation or outbreak classification.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import numpy as np
import pandas as pd

DEFAULT_SOURCE = Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\nj_daily_feature_outputs\nj_daily_features.csv.gz')
DEFAULT_OUTPUT = DEFAULT_SOURCE.parent.parent / 'dynamic_panel_nj_outputs'


def fit_within_ols(x, y):
    """x[region, training day, feature], y[region, training day]."""
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm, ym = x.mean(axis=1), y.mean(axis=1)
    xw, yw = x-xm[:,None,:], y-ym[:,None]
    scale = np.sqrt(np.mean(xw*xw, axis=(0,1)))
    scale = np.where(scale>1e-12,scale,1)
    design = (xw/scale).reshape(-1,x.shape[2])
    bscaled, _, rank, _ = np.linalg.lstsq(design,yw.reshape(-1),rcond=None)
    beta = bscaled/scale
    alpha = ym-xm@beta
    assert np.isfinite(beta).all() and np.isfinite(alpha).all()
    return beta,alpha,int(rank)


def metric_rows(frame, columns, group_columns):
    """Evaluate count predictions; each model's usable observations are explicit."""
    rows=[]
    for key,part in frame.groupby(group_columns,sort=True):
        key = key if isinstance(key,tuple) else (key,)
        for model,column in columns.items():
            good = part[column].notna() & part.actual_cases.notna()
            q=part.loc[good]
            if q.empty:
                continue
            errors=q[column].to_numpy()-q.actual_cases.to_numpy()
            rows.append({**dict(zip(group_columns,key)), 'model':model,
                'rows':len(q),'regions':q.zip.nunique(),'forecast_dates':q.date.nunique(),
                'MAE':float(np.mean(np.abs(errors))),'RMSE':float(np.sqrt(np.mean(errors**2))),
                'mean_error_prediction_minus_actual':float(np.mean(errors)),
                'mean_actual':float(q.actual_cases.mean()),
                'target_start':str(q.target_date.min().date()),'target_end':str(q.target_date.max().date())})
    return pd.DataFrame(rows)


def run_experiment(source=DEFAULT_SOURCE, output=DEFAULT_OUTPUT, window=90, retrain_every=7,
                   validation_start='2021-01-01', validation_end='2021-12-31',
                   test_start='2022-01-01', test_end='2023-12-31', include_pressure=True):
    assert window>=14 and retrain_every>=1
    source,output=Path(source),Path(output)
    assert source.is_file(), f'Cannot find {source}'
    output.mkdir(parents=True,exist_ok=True)
    run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_folder=output/f'window_{window}d_{run_id}'
    run_folder.mkdir()
    use=['zip','date','cases_today','cases_lag_1','cases_lag_7','population','valid_population_denominator']
    if include_pressure:
        use+=['nj_covid_pressure_7d','pressure_source_weight_coverage_nj_7d']
    panel=pd.read_csv(source,usecols=use,dtype={'zip':'string'},parse_dates=['date'])
    panel.zip=panel.zip.str.strip().str.zfill(5)
    assert panel.zip.str.fullmatch(r'[0-9]{5}').all() and panel.date.notna().all()
    assert not panel.duplicated(['zip','date']).any()
    ids=sorted(panel.zip.unique()); dates=pd.date_range(panel.date.min(),panel.date.max())
    assert len(panel)==len(ids)*len(dates)
    def matrix(c):
        return panel.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(dtype=float)
    counts=matrix('cases_today'); lag1=matrix('cases_lag_1'); lag7=matrix('cases_lag_7')
    assert np.isfinite(counts).all() and (counts>=0).all()
    assert np.array_equal(lag1[:,1:],counts[:,:-1])
    assert np.array_equal(lag7[:,7:],counts[:,:-7])
    n,T=counts.shape
    x_base=np.stack([counts,lag1,lag7],axis=2)
    pressure=matrix('nj_covid_pressure_7d') if include_pressure else None
    if include_pressure:
        # Eligibility is determined on the first full trailing-window day, using
        # static network/population availability, not later case outcomes.
        complete_coverage=matrix('pressure_source_weight_coverage_nj_7d')
        pressure_eligible=np.isfinite(pressure[:,6]) & np.isclose(complete_coverage[:,6],1,rtol=0,atol=1e-10)
        if pressure_eligible.any():
            assert np.isfinite(pressure[pressure_eligible,6:]).all(), 'Source coverage changes: explicitly revise the comparison cohort.'
    else:
        pressure_eligible=np.zeros(n,dtype=bool)
    horizons=[1,2,3]; max_h=max(horizons)
    vs,ve,ts,te=map(pd.Timestamp,[validation_start,validation_end,test_start,test_end])
    assert dates.min()<vs<=ve<ts<=te<=dates.max()
    config={'run_id':run_id,'training_window_calendar_days':window,'retrain_every_forecast_days':retrain_every,
        'horizons':horizons,'validation_target_boundaries':[str(vs.date()),str(ve.date())],
        'test_target_boundaries':[str(ts.date()),str(te.date())],
        'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'model':'Direct horizon-specific dynamic panel with region fixed effects, estimated by within OLS.',
        'features':['cases_today','cases_lag_1','cases_lag_7','target_weekday (Monday reference, six dummies)'],
        'pressure_variant':'Same model + nj_covid_pressure_7d on fixed source-coverage-eligible cohort' if include_pressure else 'Disabled',
        'outcome':'Recorded-onset rows after exact deduplication. Missing onset excluded, absent ZIP/date records zero as confirmed by user.',
        'evaluation':'MAE/RMSE on nonnegative-clipped count forecasts; raw predictions retained. No rounding to integers.',
        'training_boundary':'For fit date t and horizon h, 90 (or configured W) origin dates end at t-h; all training labels <= t.',
        'weekly_policy':'Fixed-effects coefficients refit weekly; each day uses newly observed history. Refit schedule restarts at each split.',
        'test_policy':'NOT EVALUATED. This experiment uses validation data only for training-window selection.',
        'geography':'ZIP/ZCTA mapping remains unvalidated; pressure conditional on supplied static population/geographic assumptions.',
        'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__},
        'limitations':['Retrospective onset-date evaluation, not real-time reporting simulation.','Fixed-effects lagged-outcome OLS is a predictive baseline, not causal inference or a GMM estimator.','Onset completeness changes substantially by year; inspect year-specific metrics.','No outbreak labels, PR-AUC, warning lead time or Moran diagnostics computed in this experiment.']}
    config['test_evaluated'] = False
    (run_folder/'experiment_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    pd.DataFrame({'zip':ids,'pressure_comparison_eligible':pressure_eligible}).to_csv(run_folder/'region_cohort.csv',index=False)
    records=[]; fit_log=[]; coefficients=[]
    # Tests compare within transformation with an explicit region-dummy OLS fit.
    rng=np.random.default_rng(21)
    sx=rng.normal(size=(3,20,2)); sy=np.array([1.,2.,3.])[:,None]+sx@np.array([.4,-.2])+rng.normal(scale=.1,size=(3,20))
    sb,sa,_=fit_within_ols(sx,sy)
    dummy=np.repeat(np.eye(3),20,axis=0)
    direct=np.linalg.lstsq(np.column_stack([dummy,sx.reshape(-1,2)]),sy.reshape(-1),rcond=None)[0]
    assert np.allclose(sa,direct[:3]) and np.allclose(sb,direct[3:])
    print(f'Regions: {n}; pressure comparison regions: {pressure_eligible.sum()}; window: {window} days.',flush=True)
    for h in horizons:
        target_weekdays=(dates+pd.Timedelta(days=h)).dayofweek.to_numpy()
        weekday=np.eye(7)[target_weekdays,1:]
        x=np.concatenate([x_base,np.broadcast_to(weekday,(n,T,6))],axis=2)
        names=['cases_today','cases_lag_1','cases_lag_7']+[f'target_weekday_{i}' for i in range(2,8)]
        for split,start,end in [('validation',vs,ve)]:
            # A common origin set across h=1,2,3 keeps all targets within the split.
            origins=np.flatnonzero((dates>=start) & ((dates+pd.Timedelta(days=max_h))<=end))
            assert len(origins)>0
            for block_start in range(0,len(origins),retrain_every):
                block=origins[block_start:block_start+retrain_every]
                fit_i=int(block[0]); train_last=fit_i-h; train_first=train_last-window+1
                assert train_first>=7, 'Not enough pre-validation history for this window.'
                train=np.arange(train_first,train_last+1)
                assert len(train)==window and (train+h).max()<=fit_i
                assert (dates[train]+pd.Timedelta(days=h)).max()<=dates[fit_i]
                y=counts[:,train+h]
                beta,alpha,rank=fit_within_ols(x[:,train,:],y)
                base_raw=np.einsum('ntp,p->nt',x[:,block,:],beta)+alpha[:,None]
                pressure_raw=np.full_like(base_raw,np.nan)
                matched_base_raw=np.full_like(base_raw,np.nan)
                models=[('Dynamic panel FE',beta,alpha,rank,n,names)]
                if include_pressure and pressure_eligible.any():
                    bm,am,rm=fit_within_ols(x[pressure_eligible][:,train,:],y[pressure_eligible])
                    matched_base_raw[pressure_eligible]=np.einsum('ntp,p->nt',x[pressure_eligible][:,block,:],bm)+am[:,None]
                    xp=np.concatenate([x[pressure_eligible],pressure[pressure_eligible,:,None]],axis=2)
                    bp,ap,rp=fit_within_ols(xp[:,train,:],y[pressure_eligible])
                    pressure_raw[pressure_eligible]=np.einsum('ntp,p->nt',xp[:,block,:],bp)+ap[:,None]
                    models.append(('Dynamic panel FE + pressure',bp,ap,rp,int(pressure_eligible.sum()),names+['nj_covid_pressure_7d']))
                    models.append(('Dynamic panel FE (matched cohort)',bm,am,rm,int(pressure_eligible.sum()),names))
                for name,b,a,r,num_regions,feature_names in models:
                    fit_log.append({'split':split,'horizon':h,'model':name,'fit_date':dates[fit_i],
                        'training_origin_start':dates[train_first],'training_origin_end':dates[train_last],
                        'latest_training_target':dates[train_last+h],
                        'prediction_origin_start':dates[block[0]],'prediction_origin_end':dates[block[-1]],
                        'training_dates':window,'training_rows':window*num_regions,'regions':num_regions,
                        'within_design_rank':r,'features':len(b),'negative_raw_predictions':int((base_raw<0).sum()) if name=='Dynamic panel FE' else int(((matched_base_raw if name=='Dynamic panel FE (matched cohort)' else pressure_raw)[pressure_eligible]<0).sum())})
                    for feature,value in zip(feature_names,b):
                        coefficients.append({'split':split,'horizon':h,'model':name,'fit_date':dates[fit_i],'feature':feature,'coefficient':float(value)})
                seasonal=counts[:,block+h-7]
                L=len(block)
                records.append(pd.DataFrame({'split':split,'zip':np.repeat(ids,L),
                    'date':np.tile(dates[block].to_numpy(),n),'target_date':np.tile(dates[block+h].to_numpy(),n),
                    'horizon':h,'fit_date':dates[fit_i],
                    'actual_cases':counts[:,block+h].reshape(-1),
                    'persistence_prediction':counts[:,block].reshape(-1),
                    'seasonal_naive_prediction':seasonal.reshape(-1),
                    'panel_prediction_raw':base_raw.reshape(-1),
                    'panel_prediction':np.maximum(base_raw,0).reshape(-1),
                    'matched_panel_prediction_raw':matched_base_raw.reshape(-1),
                    'matched_panel_prediction':np.maximum(matched_base_raw,0).reshape(-1),
                    'pressure_prediction_raw':pressure_raw.reshape(-1),
                    'pressure_prediction':np.maximum(pressure_raw,0).reshape(-1),
                    'pressure_comparison_eligible':np.repeat(pressure_eligible,L)}))
                if block_start%(retrain_every*13)==0:
                    print(f'h={h}, {split}, forecast block begins {dates[fit_i].date()}',flush=True)
    predictions=pd.concat(records,ignore_index=True)
    assert not predictions.duplicated(['zip','date','horizon']).any()
    assert (predictions.fit_date<=predictions.date).all()
    assert (predictions.target_date==predictions.date+pd.to_timedelta(predictions.horizon,unit='D')).all()
    logs=pd.DataFrame(fit_log)
    assert logs.latest_training_target.le(logs.fit_date).all()
    base_columns={'Dynamic panel FE':'panel_prediction','Persistence: future = today':'persistence_prediction','Seasonal naive: same weekday last week':'seasonal_naive_prediction'}
    metrics=metric_rows(predictions,base_columns,['split','horizon'])
    yearly=metric_rows(predictions.assign(target_year=predictions.target_date.dt.year),base_columns,['split','target_year','horizon'])
    by_region=metric_rows(predictions,base_columns,['split','zip','horizon'])
    metrics['cohort']='all_panel_regions'
    matched=pd.DataFrame()
    if include_pressure and pressure_eligible.any():
        # Every row/model in this table is scored on exactly the same observations.
        pair=predictions.loc[predictions.pressure_comparison_eligible].copy()
        assert pair.pressure_prediction.notna().all()
        matched_columns={**base_columns,'Dynamic panel FE':'matched_panel_prediction','Dynamic panel FE + pressure':'pressure_prediction'}
        matched=metric_rows(pair,matched_columns,['split','horizon'])
        matched['cohort']='pressure_eligible_regions_only'
        assert matched.groupby(['split','horizon']).rows.nunique().eq(1).all()
        matched.to_csv(run_folder/'metrics_matched_pressure_comparison.csv',index=False)
    metrics.to_csv(run_folder/'metrics_by_horizon.csv',index=False)
    yearly.to_csv(run_folder/'metrics_by_year.csv',index=False)
    by_region.to_csv(run_folder/'metrics_by_region.csv',index=False)
    logs.to_csv(run_folder/'training_log.csv',index=False)
    pd.DataFrame(coefficients).to_csv(run_folder/'feature_coefficients.csv',index=False)
    predictions.head(100).to_csv(run_folder/'prediction_preview.csv',index=False)
    print('Saving predictions and experiment records.',flush=True)
    predictions.to_csv(run_folder/'predictions.csv.gz',index=False,compression='gzip')
    details={'run_id':run_id,'rows_predictions':len(predictions),'regions':n,'pressure_comparison_regions':int(pressure_eligible.sum()),
        'model_fits':len(logs),'all_checks_passed':True,'outputs':str(run_folder.resolve()),
        'forecast_origin_dates_by_split':{k:int(g.date.nunique()) for k,g in predictions.groupby('split')},
        'selection_status':'Validation-only candidate window. Test has not been evaluated in this experiment.'}
    (run_folder/'run_summary.json').write_text(json.dumps(details,indent=2),encoding='utf-8')
    print(metrics.to_string(index=False),flush=True)
    print('Saved to:',run_folder.resolve(),flush=True)
    return predictions,metrics,matched,run_folder




def compare_windows(source=DEFAULT_SOURCE, output=None, windows=(30,60,90,180),
                    validation_start='2021-01-01', validation_end='2021-12-31', retrain_every=7):
    source=Path(source)
    if output is None:
        output=source.parent.parent/'dynamic_panel_window_comparison_outputs'
    output=Path(output)
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    session=output/f'validation_windows_{stamp}'
    session.mkdir(parents=True)
    windows=tuple(sorted(set(int(w) for w in windows)))
    assert windows and min(windows)>=14
    population=[]; all_metrics=[]; run_records=[]
    reference_keys=None; reference_actual=None
    for window in windows:
        print(f'\nComparing training window: {window} days (validation only)',flush=True)
        predictions,metrics,matched,run_folder=run_experiment(
            source,session/'candidate_runs',window=window,retrain_every=retrain_every,
            validation_start=validation_start,validation_end=validation_end,include_pressure=True)
        assert predictions.split.eq('validation').all() and matched.split.eq('validation').all()
        assert not matched.empty
        pair=predictions.loc[predictions.pressure_comparison_eligible].sort_values(['zip','date','horizon'])
        keys=pd.MultiIndex.from_frame(pair[['zip','date','horizon']])
        actual=pair.actual_cases.to_numpy()
        if reference_keys is None:
            reference_keys=keys; reference_actual=actual.copy()
        else:
            assert keys.equals(reference_keys), 'Candidate windows must use identical evaluation rows.'
            assert np.array_equal(actual,reference_actual)
        matched=matched.copy(); matched['training_window_days']=window
        all_metrics.append(matched)
        cohort=pd.read_csv(run_folder/'region_cohort.csv',dtype={'zip':str})
        population.append(cohort.loc[cohort.pressure_comparison_eligible,'zip'].tolist())
        run_records.append({'training_window_days':window,'run_folder':str(run_folder.resolve()),
                            'validation_prediction_rows_matched':len(pair),'regions_matched':pair.zip.nunique()})
        del predictions,pair
    assert all(c==population[0] for c in population)
    results=pd.concat(all_metrics,ignore_index=True)
    assert results.groupby(['training_window_days','horizon']).rows.nunique().eq(1).all()
    scores=results.loc[results.model.isin(['Dynamic panel FE','Dynamic panel FE + pressure'])].copy()
    ranked=scores.groupby(['model','training_window_days'],as_index=False).agg(
        mean_validation_MAE=('MAE','mean'),mean_validation_RMSE=('RMSE','mean'),
        horizons=('horizon','nunique'),regions=('regions','min'))
    assert ranked.horizons.eq(3).all()
    ranked=ranked.sort_values(['model','mean_validation_MAE','mean_validation_RMSE','training_window_days'])
    ranked['rank_within_model']=ranked.groupby('model').cumcount()+1
    # Primary decision declared before viewing the scores: mean MAE across h=1/2/3
    # for the pressure model. RMSE breaks ties. Prefer a shorter window only if tied.
    primary=ranked.loc[ranked.model.eq('Dynamic panel FE + pressure')].iloc[0]
    selected=int(primary.training_window_days)
    improvement=[]
    for window in windows:
        q=results.loc[results.training_window_days.eq(window)]
        base=q.loc[q.model.eq('Dynamic panel FE')].set_index('horizon')
        plus=q.loc[q.model.eq('Dynamic panel FE + pressure')].set_index('horizon')
        for h in [1,2,3]:
            improvement.append({'training_window_days':window,'horizon':h,
                'MAE_without_pressure':float(base.loc[h,'MAE']),'MAE_with_pressure':float(plus.loc[h,'MAE']),
                'MAE_reduction_percent':float((base.loc[h,'MAE']-plus.loc[h,'MAE'])/base.loc[h,'MAE']*100),
                'RMSE_without_pressure':float(base.loc[h,'RMSE']),'RMSE_with_pressure':float(plus.loc[h,'RMSE']),
                'RMSE_reduction_percent':float((base.loc[h,'RMSE']-plus.loc[h,'RMSE'])/base.loc[h,'RMSE']*100)})
    results.to_csv(session/'validation_metrics_all_windows.csv',index=False)
    ranked.to_csv(session/'window_ranking.csv',index=False)
    pd.DataFrame(improvement).to_csv(session/'pressure_improvement_by_window.csv',index=False)
    pd.DataFrame(run_records).to_csv(session/'candidate_run_index.csv',index=False)
    selection={'selected_window_days':selected,'primary_model':'Dynamic panel FE + pressure',
        'rule':'Minimum mean validation MAE over horizons 1/2/3; RMSE then shorter window break ties.',
        'mean_validation_MAE':float(primary.mean_validation_MAE),'mean_validation_RMSE':float(primary.mean_validation_RMSE),
        'candidate_windows':list(windows),'regions':len(population[0]),'same_validation_rows_verified':True,
        'validation_start':validation_start,'validation_end':validation_end,'retrain_every':retrain_every,
        'test_evaluated':False,'scope':'Conditional choice for this model, features, cohort and validation period; not a universal best window.',
        'next_step':'Freeze settings before subsequent test assessment. Earlier test results have already been inspected, so they are not an untouched new test.'}
    (session/'selected_configuration.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
    # Plain-English summary readable on GitHub without running the notebook.
    text='# Validation-only training-window comparison\n\n'
    text+=f"Selected starting window for the pressure model: **{selected} days**, using mean validation MAE across horizons 1/2/3. No test forecasts were generated.\n\n"
    text+='| Model | Window days | Mean MAE | Mean RMSE | Rank within model |\n|---|---:|---:|---:|---:|\n'
    for row in ranked.itertuples():
        text+=f'| {row.model} | {row.training_window_days} | {row.mean_validation_MAE:.6f} | {row.mean_validation_RMSE:.6f} | {row.rank_within_model} |\n'
    text+='\nBoth variants were fitted/scored on the same matched region set, and every candidate used identical validation rows. Candidate runs also save all-region reference metrics, but rankings use the matched-cohort results only. Simple benchmark predictions are unchanged across windows. The primary model is selected by a declared MAE rule; the no-pressure model ranking is also retained. Do not compare each model at a different window to isolate a pressure effect.\n'
    text+='\nThe choice is conditional on this validation period and the retrospective recorded-onset outcome. Existing onset-missingness and ZIP/ZCTA assumptions still apply. Test results inspected earlier are not newly untouched evidence.\n'
    (session/'RESULTS.md').write_text(text,encoding='utf-8')
    print('\nWindow comparison completed:',session.resolve(),flush=True)
    print(ranked.to_string(index=False),flush=True)
    return results,ranked,pd.DataFrame(improvement),selection,session



## 2. Run the window comparison

May take several minutes, including saving each candidate's predictions. Each comparison creates its own folder.

In [2]:
SOURCE = Path.cwd() / "nj_daily_feature_outputs" / "nj_daily_features.csv.gz"
OUTPUT = Path.cwd() / "dynamic_panel_window_comparison_outputs"
assert SOURCE.is_file(), "Open from Weikun or set SOURCE to the feature file."

results, ranking, pressure_effects, selected, COMPARISON_FOLDER = compare_windows(
    source=SOURCE,
    output=OUTPUT,
    windows=(30,60,90,180),
    validation_start="2021-01-01",
    validation_end="2021-12-31",
    retrain_every=7,
)



Comparing training window: 30 days (validation only)
Regions: 598; pressure comparison regions: 590; window: 30 days.
h=1, validation, forecast block begins 2021-01-01
h=1, validation, forecast block begins 2021-04-02
h=1, validation, forecast block begins 2021-07-02
h=1, validation, forecast block begins 2021-10-01
h=2, validation, forecast block begins 2021-01-01
h=2, validation, forecast block begins 2021-04-02
h=2, validation, forecast block begins 2021-07-02
h=2, validation, forecast block begins 2021-10-01
h=3, validation, forecast block begins 2021-01-01
h=3, validation, forecast block begins 2021-04-02
h=3, validation, forecast block begins 2021-07-02
h=3, validation, forecast block begins 2021-10-01
Saving predictions and experiment records.
     split  horizon                                  model   rows  regions  forecast_dates      MAE     RMSE  mean_error_prediction_minus_actual  mean_actual target_start target_end            cohort
validation        1                   

## 3. Read the rankings

Lower MAE/RMSE is better. Rankings use the same matched cohort for every model/window. No test scores appear in these tables.

In [3]:
display(ranking.round({"mean_validation_MAE":4,"mean_validation_RMSE":4}))
display(pressure_effects.round(4))
print("Selected window:", selected["selected_window_days"], "days")
print("Matched regions:", selected["regions"])
print("Saved to:", COMPARISON_FOLDER.resolve())


,model,training_window_days,mean_validation_MAE,mean_validation_RMSE,horizons,regions,rank_within_model
0,Dynamic panel FE,30,1.1335,1.9871,3,590,1
1,Dynamic panel FE,60,1.1439,1.9909,3,590,2
2,Dynamic panel FE,90,1.1580,2.0072,3,590,3
3,Dynamic panel FE,180,1.1884,2.0277,3,590,4
4,Dynamic panel FE + pressure,30,1.1273,1.9508,3,590,1
5,Dynamic panel FE + pressure,60,1.1297,1.9539,3,590,2
6,Dynamic panel FE + pressure,90,1.1352,1.9672,3,590,3
7,Dynamic panel FE + pressure,180,1.1514,1.9904,3,590,4


,training_window_days,horizon,MAE_without_pressure,MAE_with_pressure,MAE_reduction_percent,RMSE_without_pressure,RMSE_with_pressure,RMSE_reduction_percent
0,30,1,1.1052,1.0981,0.6474,1.9042,1.8685,1.8759
1,30,2,1.1365,1.1288,0.6794,1.9918,1.9543,1.8836
2,30,3,1.1588,1.1551,0.3188,2.0654,2.0298,1.7258
3,60,1,1.1127,1.0993,1.2037,1.9026,1.8681,1.8111
4,60,2,1.1464,1.1311,1.3277,1.9949,1.9578,1.8636
5,60,3,1.1727,1.1586,1.2038,2.0752,2.0359,1.8947
6,90,1,1.1240,1.1039,1.7874,1.9165,1.8798,1.9171
7,90,2,1.1604,1.1371,2.0086,2.0101,1.9703,1.9792
8,90,3,1.1896,1.1646,2.0944,2.0950,2.0515,2.0743
9,180,1,1.1570,1.1237,2.8772,1.9451,1.9100,1.8026


Selected window: 30 days
Matched regions: 590
Saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\dynamic_panel_window_comparison_outputs\validation_windows_20261026T005454220611Z


## Saved results

- `window_ranking.csv`: mean validation MAE/RMSE across horizons and ranking for each model.
- `validation_metrics_all_windows.csv`: each window/model/horizon score, including simple benchmarks.
- `pressure_improvement_by_window.csv`: pressure effect at each fixed window/horizon.
- `selected_configuration.json`: selection rule, chosen window and conditions.
- `RESULTS.md`: readable summary for teammates.
- `candidate_run_index.csv` and `candidate_runs/`: per-window configurations, training logs, validation predictions and detailed metrics.

Freeze model settings after reviewing validation. Later compare other models against the same defined baseline and matched data. Outbreak metrics are deferred until event and warning definitions exist. Clear real-data notebook previews before committing code.
